# Notebook 3 of 7: Training a Simple RNN

## Teaching AI to Write Italian Song Lyrics

**Series: Understanding AI Through Italian Music**

---

Now comes the exciting part. We are going to build our first AI model and teach it to write Italian song lyrics.

The model we will start with is called an **RNN** -- a **Recurrent Neural Network**. Think of it as a student who reads one word at a time and tries to remember what came before. It reads left to right, building up a sense of what the sentence is about, and at every step it tries to guess what word comes next.

By the end of this notebook, you will:

1. **Build** a neural network from scratch
2. **Train** it on hundreds of Italian songs
3. **Watch** its "wrongness score" decrease as it learns
4. **Generate** brand-new Italian lyrics that never existed before

Let's get started.

---

## Setup

The cell below loads all the tools we will need. You do not need to understand every line -- just run it and move on. Here is what each group does:

- **torch** is the engine that powers our neural network (think of it as the AI's brain hardware)
- **GPT2Tokenizer** is the dictionary that converts words into numbers (AI only understands numbers)
- **src modules** contain helper code we wrote for this series: the dataset loader, the RNN model, training functions, text generation, and visualization tools

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer

# Our custom modules
from src.dataset import ItalianLyricsDataset, load_lyrics
from src.models import RNNModel, model_summary
from src.training import train_model, get_optimizer
from src.generation import generate_rnn_lstm, generate_greedy
from src.visualization import plot_training_loss, visualize_tokenization

# Show charts inside the notebook
%matplotlib inline

print("All tools loaded successfully!")

## How Does an RNN Learn?

Imagine you are listening to someone read a song lyric word by word. After each word, you try to guess the next one. If you guess wrong, someone tells you the right answer and you adjust your intuition. After hearing thousands of songs, you get better at guessing.

**That is exactly what an RNN does.**

Here is a simple picture of the process. The RNN reads one word at a time, keeps a "memory" of what it has seen so far, and at each step predicts what comes next:

```
Input:    "Amore"  -->  "mio"  -->  "ti"  -->  "penso"
               |            |           |           |
RNN:      [memory]  --> [memory]  --> [memory]  --> [memory]
               |            |           |           |
Predicts:  "mio"       "ti"       "penso"     "sempre"
```

At the beginning, the RNN's guesses are completely random -- it has no idea what Italian words even look like. But every time it guesses wrong, it adjusts its internal numbers slightly to be less wrong next time. After reading hundreds of songs thousands of times, it starts to pick up patterns:

- "Amore" is often followed by "mio"
- Lines that start with "Non" often continue with "posso" or "voglio"
- Certain words tend to appear at the end of a line

This process of guessing, checking, and adjusting is called **training**.

## Load the Data

Before the RNN can learn anything, it needs data to study. We will load 500 Italian song lyrics from the file we prepared in Notebook 1. (The full dataset has over 9,000 songs, but 500 is enough for a quick experiment.)

We also need to set up:
- A **tokenizer** -- the dictionary that converts words into numbers
- A **dataset** -- packages the lyrics in a format PyTorch can work with
- A **dataloader** -- feeds the data to the model in small batches of 32 songs at a time (like giving a student a manageable stack of flashcards rather than the entire textbook at once)

In [ ]:
# Load 500 Italian song lyrics
lyrics = load_lyrics('../data/italian_lyrics.txt', max_songs=500)
print(f"Loaded {len(lyrics)} songs")

# Set up the tokenizer (word-to-number dictionary)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token  # Tell the tokenizer how to handle padding

# Create the dataset and dataloader
MAX_LENGTH = 128   # Maximum number of tokens per song
BATCH_SIZE = 32    # How many songs to process at once

dataset = ItalianLyricsDataset(lyrics, tokenizer, max_length=MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Vocabulary size: {len(tokenizer):,} tokens")
print(f"Max sequence length: {MAX_LENGTH} tokens")
print(f"Batch size: {BATCH_SIZE} songs per batch")
print(f"Number of batches per epoch: {len(dataloader)}")
print(f"\nA quick peek at the first song (first 100 characters):")
print(f"  \"{lyrics[0][:100].strip()}...\"")


## Build the Model

Now we create the RNN itself. Think of this step as building a brain -- it has the right structure but no knowledge yet. All the numbers inside it are completely random.

The model has three layers:
1. **Embedding layer** -- converts each token (number) into a richer representation of 256 numbers. This is how the model learns that "amore" and "cuore" are related concepts.
2. **RNN layer** -- reads the sequence one token at a time, updating its 512-number "memory" at each step.
3. **Output layer** -- takes the memory and produces a prediction: for each position, which of the 50,257 possible tokens is most likely to come next?

Let's build it and see how many numbers (called "parameters") are inside.

In [ ]:
# Set the device (CPU in our case)
device = torch.device('cpu')

# Model settings
VOCAB_SIZE = len(tokenizer)   # 50,257 possible tokens
EMBEDDING_DIM = 256           # Size of word representations
HIDDEN_DIM = 512              # Size of the RNN's memory

# Build the RNN model
rnn_model = RNNModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
).to(device)

# Show the model summary
total_params = model_summary(rnn_model, name="RNN")

print(f"\nThis model has {total_params / 1_000_000:.1f} million numbers inside it.")
print("Right now they are all random. Training will adjust these numbers")
print("until the model can predict Italian words.")

## Training: Watch the AI Learn

Now we are going to **train** the model. But what does that actually mean?

Training works like this:
1. Show the model a line of lyrics
2. At every word, ask it: "What comes next?"
3. Compare its guess to the real answer
4. Give it a score called **loss** -- this measures how wrong it was
5. Adjust the model's numbers slightly to be less wrong next time
6. Repeat for every song, multiple times

The **loss** is the most important number to watch:

| Loss value | What it means |
|:----------:|:--------------|
| **~10.0** | Completely lost -- random guessing among 50,000+ words |
| **~7.0** | Starting to learn -- picking up common Italian words |
| **~5.0** | Getting the hang of it -- sentences start to look Italian |
| **~3.0** | Pretty good -- recognizable phrases and patterns |
| **~0.0** | Perfect (never actually happens in practice) |

We want to watch this number **go down**. A dropping loss means the model is learning.

We will train for **3 epochs** -- meaning the model will read through all 500 songs three complete times. Think of it as a student re-reading their textbook three times before the exam.

In [ ]:
import time
import matplotlib.pyplot as plt

# Create the optimizer -- this controls how the model adjusts its numbers
# Think of it as the "teacher" that guides the learning process
optimizer = get_optimizer(rnn_model, is_transformer=False)

# Train for 3 epochs
EPOCHS = 3

print(f"Training the RNN on {len(lyrics)} songs for {EPOCHS} epochs...")
print(f"(This may take a few minutes on CPU)\n")

start = time.time()

history = train_model(
    model=rnn_model,
    dataloader=dataloader,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    is_transformer=False,
    model_name="RNN"
)

elapsed = time.time() - start
print(f"\nDone! Total training time: {elapsed / 60:.1f} minutes")

# Plot the training loss
print("\nLet's visualize how the loss changed during training:")
fig = plot_training_loss([history], title="RNN Training Loss")
fig.set_size_inches(8, 5)
plt.show()

## The Moment of Truth: Generate Lyrics

The model has been trained. Now let's see what it has learned by asking it to write lyrics.

We will try **two different methods** of generating text, and the difference is dramatic:

1. **Greedy decoding** -- always pick the single most likely next word. This sounds like a good idea, but it leads to a trap: the model gets stuck repeating the same word over and over (like "sole sole sole sole...").

2. **Sampling with variety** -- instead of always picking the top word, we let the model choose randomly from its best guesses. This produces much more interesting (and more human-like) output.

Let's see both in action.

In [ ]:
seed_text = "Amore mio"

# --- Method 1: Greedy decoding (always pick the most likely word) ---
greedy_output = generate_greedy(rnn_model, tokenizer, seed_text, max_length=50)

print("=" * 60)
print("  METHOD 1: Greedy Decoding")
print("  (Always picks the single most likely next word)")
print("=" * 60)
print()
print(f"  Seed: \"{seed_text}\"")
print(f"  Output: {greedy_output}")
print()
print("  Notice something? The model gets stuck in a loop!")
print("  This is the 'repetition trap' -- greedy decoding's weakness.")

print()
print()

# --- Method 2: Sampling with top-k and top-p filtering ---
sampled_output = generate_rnn_lstm(
    rnn_model, tokenizer, seed_text,
    max_length=80,
    temperature=0.8,
    top_k=50,
    top_p=0.9
)

print("=" * 60)
print("  METHOD 2: Sampling (with randomness)")
print("  (Chooses randomly from the best guesses)")
print("=" * 60)
print()
print(f"  Seed: \"{seed_text}\"")
print(f"  Output: {sampled_output}")
print()
print("  Much more varied! Not perfect, but it avoids the repetition trap.")
print("  The text may not make perfect sense yet, but it looks more like")
print("  Italian lyrics than a broken record.")

## What Went Right and What Went Wrong

Let's take stock of what just happened.

### What went right

- **The loss went down.** This means the model genuinely learned something. It started with completely random numbers and, after three passes through the data, it got measurably better at predicting Italian words.
- **It learned Italian-looking patterns.** Even if the output is not perfect, the generated text contains real Italian words and phrases. It did not produce English or random characters.
- **Sampling helped.** Switching from greedy decoding to randomized sampling eliminated the repetition trap and produced more interesting output.

### What went wrong

- **The text does not make much sense.** The lyrics might contain real Italian words, but the sentences often do not hold together logically. There is no coherent meaning or story.
- **It forgets the beginning.** If you look at longer outputs, the end of the text often has nothing to do with the beginning. The model loses track of its own "train of thought."

### Why? The Vanishing Gradient Problem

The RNN has a fundamental weakness: **it reads one word at a time, and its memory fades.** By the time it is 20 or 30 words into a sentence, the information about the first few words has been diluted through so many processing steps that it is effectively forgotten.

This is called the **vanishing gradient problem**. During training, the signal that tells the model "hey, word #1 matters for word #30" gets weaker and weaker as it passes backwards through the chain of memory steps. It is like a game of telephone -- the message gets garbled over distance.

```
Word 1  -->  Word 2  -->  Word 3  -->  ...  -->  Word 30
  |            |            |                        |
  Strong       OK           Weak         ...      Almost gone
  memory       memory       memory                 memory of Word 1
```

This is not a bug we can fix with more training. It is a fundamental limitation of the RNN architecture itself. To solve it, we need a smarter kind of memory -- and that is exactly what the next notebook is about.

## Try It Yourself

Now it is your turn! Change the `seed_text` below to any Italian phrase you like and run the cell to see what the RNN generates. Here are some suggestions to get you started:

- `"Nel blu dipinto di blu"` -- the famous Volare opening
- `"La vita e"` -- "Life is..."
- `"Quando la notte"` -- "When the night..."
- `"Non posso vivere"` -- "I cannot live..."
- `"Ti amo"` -- "I love you"

Try different phrases and see how the starting words influence what the model writes!

In [ ]:
# ============================================================
#  CHANGE THIS to any Italian phrase you like!
# ============================================================
seed_text = "La vita e"

# Generate lyrics with the RNN
generated = generate_rnn_lstm(
    rnn_model, tokenizer, seed_text,
    max_length=80,
    temperature=0.8,
    top_k=50,
    top_p=0.9
)

print(f"Seed:   \"{seed_text}\"")
print(f"Output: {generated}")
print()
print("Try changing seed_text above and running this cell again!")

## Key Takeaways

Here is what we learned in this notebook:

1. **Training = predict the next word + learn from mistakes.** The entire training process boils down to: guess the next word, see how wrong you were, adjust your numbers, repeat. This is the same basic idea behind ChatGPT, just at a much smaller scale.

2. **Loss measures how wrong the model is.** A high loss means the model is guessing randomly. A low loss means it has learned patterns. Watching the loss decrease is how we know learning is happening.

3. **RNNs learn, but they have weak memory.** Our RNN successfully learned Italian word patterns, but it cannot remember the beginning of a sentence by the time it reaches the end. This is the vanishing gradient problem.

4. **Greedy decoding causes repetition.** Always picking the single most likely next word leads to the model getting stuck in loops (like "sole sole sole..."). Adding randomness through sampling produces much more interesting and varied text.

5. **The architecture matters.** The quality of the generated text is limited not just by how much data or training time we use, but by the fundamental design of the model itself. A better architecture can learn better -- even with the same data.

## What's Next

The RNN's weak memory is a real problem. It means the model can never write coherent text that is more than a few words long, no matter how much data we give it or how long we train it.

In the next notebook, we will meet the **LSTM** -- **Long Short-Term Memory** -- a model designed specifically to remember better. It solves the vanishing gradient problem by adding clever "gates" that control what information to keep and what to throw away. Think of it as upgrading from a student with a bad memory to one who takes careful notes.

**Next up: [Notebook 4 -- LSTM: A Model That Remembers](04_lstm_better_memory.ipynb)**

---

*Notebook 3 of 7 in the "Understanding AI Through Italian Music" series.*